# Time Series Aggregation

## Overview
Time series aggregation converts individual timestamped events into regular period summaries and enables period-over-period comparisons. The core challenge is that raw transactional data has gaps -- months with no activity are simply absent from a GROUP BY result.

**Key patterns covered:**

| Pattern | Purpose |
|---|---|
| Date spine | Generate all calendar periods; LEFT JOIN to fill gaps with zeros |
| Moving average | Smooth out noise; reveal underlying trend |
| Cumulative sum | Running total over time |
| YoY / MoM comparison | Period-over-period growth using LAG with PARTITION BY |
| Period-to-date | MTD, QTD, YTD aggregations |

**Date spine approaches:**
- **SQLite:** Recursive CTE generating months one at a time
- **PostgreSQL:** `GENERATE_SERIES(start, end, interval)` -- one line, no recursion needed

```sql
-- PostgreSQL date spine (month granularity):
SELECT DATE_TRUNC('month', gs)::DATE AS month
FROM   GENERATE_SERIES('2023-01-01'::DATE, '2023-12-01'::DATE, '1 month') AS gs;
```

**PostgreSQL notes:** `DATE_TRUNC`, `GENERATE_SERIES`, and `INTERVAL` arithmetic are the standard tools. SQLite equivalents use `STRFTIME`, recursive CTEs, and `DATE(date, '+1 month')`.

---

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, segment TEXT, province TEXT, signup_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, opened_date TEXT, status TEXT, credit_limit REAL);CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, customer_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT);INSERT INTO customers VALUES (1,'Aroha','Ngata','Retail','NB','2023-01-05',1),(2,'Liam','Chen','Wealth','NS','2023-01-12',1),(3,'Fatima','Al-Rashid','SME','ON','2023-01-20',1),(4,'James','MacLeod','Retail','NB','2023-02-03',1),(5,'Sofia','Petrov','Retail','BC','2023-02-14',1),(6,'Noah','Williams','SME','AB','2023-02-28',0),(7,'Mei','Zhang','Wealth','ON','2023-03-10',1),(8,'Ethan','Tremblay','Retail','QC','2023-04-05',1),(9,'Priya','Nair','Wealth','BC','2023-04-18',1),(10,'Marcus','Okafor','SME','ON','2023-04-25',1),(11,'Diana','Flores','Retail','NB','2023-05-02',1),(12,'Ravi','Patel','Retail','ON','2023-05-15',1),(13,'Yuki','Tanaka','Wealth','BC','2023-06-01',1),(14,'Omar','Hassan','SME','QC','2023-06-20',1),(15,'Chloe','Bouchard','Retail','QC','2023-07-08',0);INSERT INTO accounts VALUES (101,1,'Chequing','2023-01-05','Active',NULL),(102,1,'Savings','2023-01-05','Active',NULL),(103,2,'Investment','2023-01-12','Active',50000),(104,3,'Chequing','2023-01-20','Active',NULL),(105,3,'Loan','2023-01-20','Active',25000),(106,4,'Chequing','2023-02-03','Active',NULL),(107,5,'Chequing','2023-02-14','Active',NULL),(108,5,'Savings','2023-02-14','Active',NULL),(109,6,'Chequing','2023-02-28','Closed',NULL),(110,7,'Investment','2023-03-10','Active',75000),(111,8,'Chequing','2023-04-05','Active',NULL),(112,9,'Investment','2023-04-18','Active',100000),(113,10,'Chequing','2023-04-25','Active',NULL),(114,10,'Loan','2023-04-25','Active',15000),(115,11,'Chequing','2023-05-02','Active',NULL),(116,12,'Chequing','2023-05-15','Active',NULL),(117,12,'Savings','2023-05-15','Active',NULL),(118,13,'Investment','2023-06-01','Active',60000),(119,14,'Chequing','2023-06-20','Active',NULL),(120,15,'Chequing','2023-07-08','Closed',NULL);INSERT INTO transactions VALUES (1001,101,1,'2023-01-10','Deposit',2500.00,'Payroll'),(1002,101,1,'2023-01-22','Withdrawal',-800.00,'Rent'),(1003,102,1,'2023-02-05','Deposit',500.00,'Transfer'),(1004,103,2,'2023-02-10','Deposit',10000.00,'Investment'),(1005,104,3,'2023-02-15','Deposit',3200.00,'Payroll'),(1006,104,3,'2023-03-01','Withdrawal',-1200.00,'Rent'),(1007,106,4,'2023-03-10','Deposit',2800.00,'Payroll'),(1008,107,5,'2023-03-15','Deposit',2200.00,'Payroll'),(1009,107,5,'2023-03-28','Fee',-25.00,'Monthly Fee'),(1010,101,1,'2023-03-10','Deposit',2500.00,'Payroll'),(1011,103,2,'2023-04-15','Deposit',15000.00,'Investment'),(1012,110,7,'2023-04-20','Deposit',20000.00,'Investment'),(1013,111,8,'2023-05-01','Deposit',2100.00,'Payroll'),(1014,112,9,'2023-05-10','Deposit',25000.00,'Investment'),(1015,113,10,'2023-05-20','Deposit',3500.00,'Payroll'),(1016,115,11,'2023-06-01','Deposit',2000.00,'Payroll'),(1017,116,12,'2023-06-15','Deposit',2700.00,'Payroll'),(1018,118,13,'2023-07-01','Deposit',18000.00,'Investment'),(1019,101,1,'2023-07-10','Deposit',2500.00,'Payroll'),(1020,103,2,'2023-07-20','Deposit',12000.00,'Investment'),(1021,104,3,'2023-07-15','Deposit',3200.00,'Payroll'),(1022,107,5,'2023-08-01','Deposit',2200.00,'Payroll'),(1023,111,8,'2023-08-05','Deposit',2100.00,'Payroll'),(1024,113,10,'2023-08-15','Withdrawal',-500.00,'Transfer'),(1025,116,12,'2023-08-20','Deposit',2700.00,'Payroll'),(1026,101,1,'2023-10-10','Deposit',2500.00,'Payroll'),(1027,116,12,'2023-10-20','Deposit',2700.00,'Payroll');""")
conn.commit()
print("Finance schema ready.")
for t in ["customers","accounts","transactions"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

---
## Date spine: filling gaps in monthly data

In [ ]:
import pandas as pd
q = """
WITH RECURSIVE spine(month) AS (
    SELECT '2023-01'
    UNION ALL
    SELECT STRFTIME('%Y-%m', DATE(month || '-01', '+1 month'))
    FROM   spine
    WHERE  month < '2023-10'
),
monthly_totals AS (
    SELECT  STRFTIME('%Y-%m', txn_date)   AS month,
            ROUND(SUM(CASE WHEN txn_type='Deposit' THEN amount ELSE 0 END), 2)  AS deposits,
            COUNT(DISTINCT customer_id)   AS active_customers
    FROM    transactions
    GROUP BY STRFTIME('%Y-%m', txn_date)
)
SELECT  s.month,
        COALESCE(m.deposits,          0)  AS deposits,
        COALESCE(m.active_customers,  0)  AS active_customers
FROM    spine       AS s
LEFT JOIN monthly_totals AS m ON s.month = m.month
ORDER BY s.month
"""
print("=== Date spine: all months Jan-Oct 2023, gaps filled with 0 ===")
df = pd.read_sql(q, conn)
print(df.to_string(index=False))
print()
gaps = (df["deposits"] == 0).sum()
print(f"Months with no deposit activity: {gaps}")
print()
print("PostgreSQL equivalent (no recursion needed):")
print("""
  SELECT DATE_TRUNC('month', gs)::DATE AS month,
         COALESCE(SUM(t.amount), 0)     AS deposits
  FROM   GENERATE_SERIES(
             '2023-01-01'::DATE,
             '2023-10-01'::DATE,
             '1 month'::INTERVAL
         ) AS gs
  LEFT JOIN transactions t
         ON DATE_TRUNC('month', t.txn_date::DATE) = DATE_TRUNC('month', gs)
        AND t.txn_type = 'Deposit'
  GROUP BY 1 ORDER BY 1;
""")

---
## Moving averages and cumulative totals

In [ ]:
import pandas as pd
q = """
WITH monthly AS (
    SELECT  STRFTIME('%Y-%m', txn_date)  AS month,
            ROUND(SUM(CASE WHEN txn_type='Deposit' THEN amount ELSE 0 END), 2) AS deposits
    FROM    transactions
    GROUP BY STRFTIME('%Y-%m', txn_date)
)
SELECT  month,
        deposits,
        ROUND(AVG(deposits) OVER (
            ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2)  AS ma3_deposits,
        ROUND(SUM(deposits) OVER (ORDER BY month ROWS UNBOUNDED PRECEDING), 2) AS cum_deposits
FROM    monthly
ORDER BY month
"""
print("=== 3-month moving average and cumulative deposits ===")
print(pd.read_sql(q, conn).to_string(index=False))
print()
print("ma3 = average of current month + 2 prior months (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)")
print("Months with fewer than 3 prior periods use available rows (window shrinks at boundaries).")

---
## Year-over-year and month-over-month comparisons

In [ ]:
import pandas as pd
q = """
WITH monthly AS (
    SELECT  STRFTIME('%Y-%m', txn_date)  AS month,
            STRFTIME('%Y', txn_date)     AS yr,
            STRFTIME('%m', txn_date)     AS mo,
            ROUND(SUM(CASE WHEN txn_type='Deposit' THEN amount ELSE 0 END), 2) AS deposits
    FROM    transactions
    GROUP BY STRFTIME('%Y-%m', txn_date)
)
SELECT  a.month,
        a.deposits                         AS curr_deposits,
        LAG(a.deposits) OVER (
            PARTITION BY a.mo ORDER BY a.yr
        )                                  AS prev_yr_deposits,
        ROUND(
            100.0 * (a.deposits -
                     LAG(a.deposits) OVER (PARTITION BY a.mo ORDER BY a.yr))
            / NULLIF(LAG(a.deposits) OVER (PARTITION BY a.mo ORDER BY a.yr), 0),
        1)  AS yoy_pct_change
FROM    monthly AS a
ORDER BY a.month
"""
print("=== YoY deposit growth (LAG partitioned by calendar month) ===")
print(pd.read_sql(q, conn).to_string(index=False))
print()
print("MoM comparison: remove PARTITION BY mo; LAG looks at the immediately prior month.")
print()
print("Period-to-date (MTD/QTD/YTD) pattern in PostgreSQL:")
print("""
  -- Month-to-date deposits as of today:
  SELECT SUM(amount)
  FROM   transactions
  WHERE  txn_type = 'Deposit'
    AND  txn_date >= DATE_TRUNC('month', CURRENT_DATE)
    AND  txn_date <  CURRENT_DATE + INTERVAL '1 day';

  -- Quarter-to-date:
  WHERE txn_date >= DATE_TRUNC('quarter', CURRENT_DATE)

  -- Year-to-date:
  WHERE txn_date >= DATE_TRUNC('year', CURRENT_DATE);
""")
conn.close()

---
## Common Pitfalls

**1. Missing months due to no LEFT JOIN to a date spine**
`GROUP BY STRFTIME('%Y-%m', txn_date)` produces only months that appear in the data. September 2023 with zero deposits simply does not appear. Charts and moving averages built on this data have implicit gaps. Always LEFT JOIN from a date spine to guarantee every period appears.

**2. Moving average window shrinking silently at boundaries**
`ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` uses 1 or 2 rows at the start of the series (not 3), producing a 1-month and 2-month average for the first two periods. This is usually correct but should be documented. If you need a minimum window size, filter out early periods or use `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` with a NULL guard.

**3. YoY LAG without PARTITION BY month produces MoM instead**
`LAG(deposits) OVER (ORDER BY month)` compares to the immediately prior row -- which is the prior month, not the same month last year. For year-over-year, use `LAG(deposits) OVER (PARTITION BY STRFTIME('%m', month) ORDER BY STRFTIME('%Y', month))` to compare same calendar month across years.

**4. Division by zero in growth rate calculations**
`100.0 * (curr - prev) / prev` crashes when prev is 0 (first deposit month, or a zero-revenue month). Always wrap the denominator: `NULLIF(prev, 0)`. This returns NULL for undefined growth rates rather than an error.

**5. GENERATE_SERIES in PostgreSQL only works with correct interval casting**
`GENERATE_SERIES('2023-01-01', '2023-12-01', '1 month')` requires the interval `'1 month'` to be cast as `::INTERVAL`. Without the cast, PostgreSQL may interpret it as a string comparison and generate unexpected results or an error.

**6. Aggregating before and after timezone conversion gives different results**
`DATE_TRUNC('day', txn_timestamp)` in UTC groups differently from `DATE_TRUNC('day', txn_timestamp AT TIME ZONE 'America/Toronto')`. A transaction at 23:30 UTC is midnight the next day in ET. Always convert to the business timezone before truncating to periods.


---
*sql_methods_library - Samantha McGarrigle*